[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/05-cnns-classification.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/05-cnns-classification.ipynb)

# Module 6.5 — CNNs and Image Classification
**Module 6: Computer Vision** | Estimated time: 50 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Explain the CNN architecture: convolution, pooling, and fully-connected layers
- Build a simple CNN from scratch using `torch.nn`
- Apply transfer learning with ResNet18 from `torchvision.models`
- Freeze base layers and replace the final classification head
- Use `torchvision.transforms` for data augmentation
- Train on a CIFAR-10 subset and plot training / validation curves

In [ ]:
# PyTorch ships pre-installed on Colab — just import
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')
print(f'torchvision {torchvision.__version__}')

# CIFAR-10 class names
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

torch.manual_seed(42)
np.random.seed(42)

## CNN Architecture Basics

A Convolutional Neural Network processes images through a series of layers:

```
Input Image (3×32×32)
    │
    ▼
Conv2d(3→32, 3×3) + ReLU   →  32×30×30
MaxPool2d(2×2)              →  32×15×15
    │
    ▼
Conv2d(32→64, 3×3) + ReLU  →  64×13×13
MaxPool2d(2×2)              →  64×6×6
    │
    ▼
Flatten                     →  2304
Linear(2304→256) + ReLU
Dropout(0.5)
Linear(256→10)              →  10 class logits
```

**Conv2d**: learns spatial filters; detects edges, textures, shapes.  
**MaxPool2d**: downsamples, adds spatial invariance.  
**Linear**: makes the final classification decision.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3×32×32 → 32×32×32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                           # → 32×16×16

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # → 64×16×16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                           # → 64×8×8

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),# → 128×8×8
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                           # → 128×4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn = SimpleCNN(num_classes=10)
cnn = cnn.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in cnn.parameters())
train_params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {train_params:,}')

# Test forward pass
dummy = torch.randn(4, 3, 32, 32).to(DEVICE)
out = cnn(dummy)
print(f'Input shape:  {dummy.shape}')
print(f'Output shape: {out.shape}  (batch=4, classes=10)')

## Data Augmentation with torchvision.transforms

Data augmentation artificially expands the training dataset by applying random transformations. This reduces overfitting and improves generalisation.

Common augmentations for image classification:
- `RandomHorizontalFlip` — randomly mirror the image
- `RandomCrop` — crop to a random location
- `ColorJitter` — randomly adjust brightness, contrast, saturation, and hue
- `RandomRotation` — rotate by a small angle
- `Normalize` — subtract mean and divide by std (per channel)

In [ ]:
# CIFAR-10 mean and std (pre-computed on training set)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

print('Training augmentation pipeline:')
for t in train_transform.transforms:
    print(f'  {t}')

# Download CIFAR-10 (downloads ~170 MB on first run)
print('\nDownloading CIFAR-10 (may take a moment)...')
train_full = torchvision.datasets.CIFAR10(
    root='/tmp/cifar10', train=True,  download=True, transform=train_transform)
val_full   = torchvision.datasets.CIFAR10(
    root='/tmp/cifar10', train=False, download=True, transform=val_transform)
print(f'Training samples: {len(train_full):,}')
print(f'Validation samples: {len(val_full):,}')

## Visualising Augmented Samples

Let us visualise a batch of CIFAR-10 images before and after augmentation to understand what the transforms actually do to the data.

In [ ]:
# Show some sample images from CIFAR-10 (before normalisation for visibility)
view_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])
train_view = torchvision.datasets.CIFAR10(
    root='/tmp/cifar10', train=True, download=False, transform=view_transform)

loader = DataLoader(train_view, batch_size=32, shuffle=True)
images, labels = next(iter(loader))

def denorm(tensor):
    return (tensor.numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for ax, img, lbl in zip(axes.flat, images[:32], labels[:32]):
    ax.imshow(denorm(img))
    ax.set_title(CLASSES[lbl], fontsize=6)
    ax.axis('off')
plt.suptitle('CIFAR-10 — Sample Training Images (with augmentation)', fontweight='bold')
plt.tight_layout(); plt.show()

## Transfer Learning with ResNet18

**Transfer learning** leverages features learned on a large dataset (ImageNet, 1.2M images, 1000 classes) for a new, smaller task.

Strategy:
1. Load ResNet18 with pre-trained ImageNet weights
2. **Freeze** all layers (they already know good features)
3. **Replace** the final fully-connected layer with one matching our number of classes
4. Only the new FC layer is trained — much faster, needs less data

In [ ]:
# Load ResNet18 with ImageNet weights
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all parameters
for param in resnet.parameters():
    param.requires_grad = False

# Replace final FC layer (was 512→1000, now 512→10 for CIFAR-10)
in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, 10)  # only this layer is trainable
resnet = resnet.to(DEVICE)

# Count frozen vs. trainable params
frozen   = sum(p.numel() for p in resnet.parameters() if not p.requires_grad)
trained  = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'Frozen parameters   : {frozen:,}  (pre-trained backbone)')
print(f'Trainable parameters: {trained:,}  (new classification head only)')
print(f'Total parameters    : {frozen+trained:,}')

# Verify only fc is trainable
for name, param in resnet.named_parameters():
    if param.requires_grad:
        print(f'  Training: {name}  shape={list(param.shape)}')

## Training Loop

We train on a small subset of CIFAR-10 (5,000 samples) for 5 epochs so the notebook runs in a reasonable time. In a real project you would use the full dataset and more epochs.

In [ ]:
# Use a small subset to keep runtime short
SUBSET_SIZE = 5000
VAL_SIZE    = 1000

idx_train = list(range(SUBSET_SIZE))
idx_val   = list(range(VAL_SIZE))
train_sub = Subset(train_full, idx_train)
val_sub   = Subset(val_full,   idx_val)

train_loader = DataLoader(train_sub, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_sub,   batch_size=64, shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

NEPOCHS = 5
print(f'Training ResNet18 (frozen backbone) on {SUBSET_SIZE} samples')
print(f'for {NEPOCHS} epochs...\n')

for epoch in range(NEPOCHS):
    t0 = time.time()

    # --- Training phase ---
    resnet.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validation phase ---
    resnet.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            v_loss    += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            v_correct += predicted.eq(labels).sum().item()
            v_total   += labels.size(0)
    val_loss = v_loss / v_total
    val_acc  = v_correct / v_total

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(f'Epoch {epoch+1}/{NEPOCHS}  '
          f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
          f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}  '
          f'({elapsed:.1f}s)')

print('\nTraining complete!')

## Plotting Training Curves

Training curves show how loss and accuracy evolve over epochs. Key things to look for:
- **Overfitting**: training accuracy much higher than validation accuracy
- **Underfitting**: both accuracies plateau at a low value
- **Good fit**: both curves improve together and converge

In [ ]:
epochs = range(1, NEPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train loss')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train acc')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curves'); axes[1].legend(); axes[1].grid(True)
axes[1].set_ylim(0, 100)

plt.suptitle('ResNet18 Transfer Learning — Training Curves', fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Final validation accuracy: {history["val_acc"][-1]*100:.1f}%')
print('Note: Transfer learning achieves ~50-70% on a 5K subset in 5 epochs.')
print('Full CIFAR-10 + 30 epochs typically reaches 90%+ with ResNet18.')

## Summary

| Concept | Key takeaway |
|---|---|
| CNN layers | Conv (feature extraction) → Pool (downsize) → FC (classify) |
| BatchNorm | Stabilises training, allows higher learning rates |
| Dropout | Regularisation — randomly zeros units during training |
| Transfer learning | Freeze backbone, train only the new head |
| Data augmentation | Random flips/crops/colour jitter reduce overfitting |
| Learning rate scheduler | Reduce LR over time for better convergence |

## Practice Exercises

**Exercise 1 — Fine-tuning vs Feature Extraction:**  
Unfreeze the last ResNet18 block (`layer4`) in addition to the FC layer. Retrain for 5 epochs with a much smaller learning rate (1e-4) for the backbone layers using two parameter groups in the optimizer. Compare accuracy vs. the frozen-backbone approach.

**Exercise 2 — Learning Rate Sensitivity:**  
Train the simple `SimpleCNN` from scratch on the same 5K CIFAR-10 subset using three learning rates: `1e-2`, `1e-3`, and `1e-4`. Plot all three loss curves on the same graph. Which converges fastest and which converges best?

**Exercise 3 — Confusion Matrix:**  
After training, run the model on the full validation set and collect all predictions and true labels. Use `sklearn.metrics.confusion_matrix` and `seaborn.heatmap` to plot the confusion matrix. Which CIFAR-10 classes are most frequently confused with each other?